# Time Complexity Comparisons

Each section solves one problem in two or three ways and times the solutions on the same inputs, to show how the running time of algorithms with different time complexities grows with the input size `n`.

| Section | Slower solution | Faster solution |
|---|---|---|
| Pairs with a sum divisible by `k` | nested loops, O(n²) | remainder counting, O(n) on average |
| Fibonacci numbers | plain recursion, Θ(φⁿ) ≈ Θ(1.618ⁿ) | memoization, O(n) |
| Membership lookups | `in` on a list, O(n) | `in` on a set, O(1) on average |
| Searching a sorted array | linear search, O(n) | binary search, O(log n) |
| Sorting | insertion sort, O(n²) | merge sort and Python's `sorted()`, O(n log n) |

**How timings are measured:** every random input is generated after `r.seed(42)`, so the inputs and answers are the same on every run. Each timing is the median of 5 repeated runs after a warm-up (see `median_time` below).

**All stored timings are machine-dependent, illustrative measurements.** They will differ on other computers and from run to run. Compare the solutions within a section, and how each solution's time changes as `n` grows, rather than the absolute numbers or times from different sections.

In [1]:
import timeit
import random as r 



In [2]:
import statistics

# Returns the median time of one call to func(*args), in seconds.
# Timer.autorange() works as a warm-up and picks how many calls each timed run makes (at least 0.2s per run),
# then the median of 5 timed runs is used so an occasional slow run does not skew the result.
def median_time(func, *args, repeat=5):
    timer = timeit.Timer(lambda: func(*args))
    number, _ = timer.autorange()
    return statistics.median(timer.repeat(repeat=repeat, number=number)) / number

# formats a time in seconds using a readable unit, e.g. 0.00123 -> "1.23 ms"
def format_time(seconds):
    for unit, scale in [("s", 1), ("ms", 1e-3), ("µs", 1e-6)]:
        if seconds >= scale:
            return f"{seconds / scale:.3g} {unit}"
    return f"{seconds / 1e-9:.3g} ns"

## O(n²) vs O(n): counting pairs with a sum divisible by `k`

The question asks how many pairs of numbers in an array `a` (each pair counted once) have a sum that is evenly divisible by a number `k`.

- **Nested loops, O(n²):** checks every pair, about n²/2 checks.
- **Remainder counting, O(n):** makes one pass through the array. For each number it looks up how many earlier numbers have the remainder that would make the pair's sum divisible by `k`. Dictionary lookups are O(1) on average, so the pass is O(n) on average and uses O(k) extra space.

In [3]:
# solution with O(n^2) time complexity 
def slow_solution(a, k):
    ans = 0 
    for i in range(0, len(a)):
        for n in range(1+i, len(a)):
            if (a[i] + a[n]) % k == 0: ans += 1
            else: pass
    return ans



In [4]:
# solution with O(n) time complexity 
def faster_solution(a, k):
    remainder_count = {}
    ans = 0

    for num in a:
        remainder = num % k
        complement = (k - remainder) % k
        ans += remainder_count.get(complement, 0)
        remainder_count[remainder] = remainder_count.get(remainder, 0) + 1
        
    return ans



In [5]:
# generating random numbers for array and number to divide by, with a fixed seed so the results are reproducible
r.seed(42)
a = [r.randint(1, 100000000) for i in range(5000)]
k = r.randint(1, 50)

In [6]:
print(f"First 15 numbers of array a:\n{a[0:15]}")
print(f"How many pairs of these numbers have a sum divisible by {k}?")

First 15 numbers of array a:
[85822413, 14942604, 3356887, 99529224, 36913811, 32868829, 29958839, 18728464, 98847494, 13756670, 90825068, 99410742, 73197858, 11668733, 79254564]
How many pairs of these numbers have a sum divisible by 23?


In [7]:
# median time of one call to each solution
time_faster_solution = median_time(faster_solution, a, k)
time_nested_loop = median_time(slow_solution, a, k)

In [8]:
print("Illustrative, machine-dependent timings (median time per call):")
print(f"Optimized Solution, O(n): \nAnswer: {faster_solution(a, k)}    Time: {format_time(time_faster_solution)}")
print(f"\nNested For Loop Solution, O(n^2): \nAnswer: {slow_solution(a, k)}    Time: {format_time(time_nested_loop)}")

Illustrative, machine-dependent timings (median time per call):
Optimized Solution, O(n): 
Answer: 543612    Time: 488 µs



Nested For Loop Solution, O(n^2): 
Answer: 543612    Time: 597 ms


## Θ(φⁿ) vs O(n): Fibonacci numbers

- **Plain recursion, Θ(φⁿ):** `fib_no_memo(n)` calls itself for `n - 1` and `n - 2`, so the same Fibonacci numbers are recalculated over and over. It makes 2·F(n+1) − 1 calls, which grows as Θ(φⁿ) ≈ Θ(1.618ⁿ), where φ = (1 + √5) / 2 is the golden ratio. O(2ⁿ) is often quoted as a simpler upper bound.
- **Memoization, O(n):** `fib_memo(n)` saves each Fibonacci number the first time it is calculated, so each number from 0 to `n` is calculated only once. This takes O(n) time and O(n) space for the saved numbers and the recursion.

Increasing `n` by 5 multiplies the plain recursion's time by about φ⁵ ≈ 11, while the memoized time grows only slightly. `n` is kept at 30 or below so the plain recursion finishes in well under a second.

In [9]:
# solution with Θ(φ^n) ≈ Θ(1.618^n) time complexity (O(2^n) is a simpler upper bound), the same Fibonacci numbers are recalculated over and over
def fib_no_memo(n):
    if n <= 1:
        return n
    return fib_no_memo(n-1) + fib_no_memo(n-2)

# solution with O(n) time and O(n) space, each Fibonacci number is calculated once and saved in "memo"
# a new memo is started on every call, so repeated timing runs do not reuse saved answers
def fib_memo(n, memo=None):
    if memo is None:
        memo = {0: 0, 1: 1}
    if n in memo:
        return memo[n]
    memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

print("Illustrative, machine-dependent timings (median time per call):")
for n in [20, 25, 30]:
    assert fib_no_memo(n) == fib_memo(n)
    time_no_memo = median_time(fib_no_memo, n)
    time_memo = median_time(fib_memo, n)
    print(f"n = {n}:  F(n) = {fib_memo(n):,}    without memoization {format_time(time_no_memo)}    with memoization {format_time(time_memo)}")

Illustrative, machine-dependent timings (median time per call):


n = 20:  F(n) = 6,765    without memoization 675 µs    with memoization 3.25 µs


n = 25:  F(n) = 75,025    without memoization 9.99 ms    with memoization 4.08 µs


n = 30:  F(n) = 832,040    without memoization 88.2 ms    with memoization 3.58 µs


## O(1) vs O(n): membership lookups with a set and a list

The question asks whether a number `q` is in a collection of `n` numbers. The same code, `q in container`, is used for both data structures:

- **List, O(n):** Python compares `q` with the items one at a time. A number that is not in the list is compared with all `n` items (the worst case), and a number that is in the list is found after about `n / 2` comparisons on average.
- **Set, O(1) on average:** Python hashes `q` and checks only the matching slots of a hash table, so the time does not depend on `n`. This is an average-case result: if many values collided in the same slots a lookup could take O(n) in the worst case, but that is very unlikely for ordinary inputs like these.

Building the set from the list costs O(n) once, so it is timed separately. Half of the 200 queries are in the collection and half are not, and the time per lookup includes the small fixed cost of the loop around it.

As `n` grows 10×, a list lookup makes 10× as many comparisons, so its time should grow at least 10×, while the set lookup time stays roughly flat. At the largest size the measured list time can grow much more than 10×: the number objects are scattered across memory, and with this many of them each comparison more often waits on main memory instead of the CPU's caches. This is a wall-clock effect that Big O does not describe; the number of comparisons still only grows 10×.

In [10]:
# counts how many of the queries are in the container, the same code is used for a list and a set
def count_present(container, queries):
    ans = 0
    for q in queries:
        if q in container: ans += 1
    return ans

In [11]:
# fixed seed so the inputs are reproducible
r.seed(42)

print("Illustrative, machine-dependent timings (median time per lookup):")
for n in [1000, 10000, 100000]:
    # n unique numbers, 100 queries that are in them and 100 that are not
    nums_list = r.sample(range(10 * n), n)
    nums_set = set(nums_list)
    queries = r.sample(nums_list, 100) + [r.randint(10 * n, 20 * n) for i in range(100)]
    assert count_present(nums_list, queries) == count_present(nums_set, queries) == 100

    time_list = median_time(count_present, nums_list, queries) / len(queries)
    time_set = median_time(count_present, nums_set, queries) / len(queries)
    time_build = median_time(set, nums_list)
    print(f"n = {n:>7,}:  list {format_time(time_list)}    set {format_time(time_set)}    (building the set once: {format_time(time_build)})")

Illustrative, machine-dependent timings (median time per lookup):


n =   1,000:  list 3.7 µs    set 19.5 ns    (building the set once: 10.4 µs)


n =  10,000:  list 42.2 µs    set 19.5 ns    (building the set once: 145 µs)


n = 100,000:  list 1.41 ms    set 21.9 ns    (building the set once: 8 ms)


## O(log n) vs O(n): searching a sorted array

The question asks for the position of a number `t` in a sorted array `s`:

- **Linear search, O(n):** checks the numbers one at a time from the start. For a target that is in the array this takes about `n / 2` checks on average and `n` checks in the worst case.
- **Binary search, O(log n):** checks the middle number and throws away the half of the array that cannot contain `t`, so it needs at most ⌊log₂ n⌋ + 1 checks, even in the worst case. It only works because the array is sorted. Python's `bisect` module provides the same algorithm.

Each size uses 100 targets taken from the array. As `n` grows 10×, linear search makes about 10× as many checks, while binary search needs only 3 or 4 more steps. As in the lookup section, the measured linear search time can grow more than 10× at the largest size because of how the numbers are laid out in memory.

In [12]:
# solution with O(n) time complexity
def linear_search(s, t):
    for i in range(len(s)):
        if s[i] == t:
            return i
    return -1

# solution with O(log n) time complexity
def binary_search(s, t):
    low, high = 0, len(s) - 1
    while low <= high:
        mid = (low + high) // 2
        if s[mid] == t:
            return mid
        elif s[mid] < t:
            low = mid + 1
        else:
            high = mid - 1
    return -1

In [13]:
# fixed seed so the inputs are reproducible
r.seed(42)

print("Illustrative, machine-dependent timings (median time per search):")
for n in [1000, 10000, 100000]:
    # a sorted array of n unique numbers and 100 targets taken from it
    s = sorted(r.sample(range(10 * n), n))
    targets = r.sample(s, 100)
    assert all(linear_search(s, t) == binary_search(s, t) == s.index(t) for t in targets)

    time_linear = median_time(lambda: [linear_search(s, t) for t in targets]) / len(targets)
    time_binary = median_time(lambda: [binary_search(s, t) for t in targets]) / len(targets)
    print(f"n = {n:>7,}:  linear {format_time(time_linear)}    binary {format_time(time_binary)}    (binary search worst case: {n.bit_length()} steps)")

Illustrative, machine-dependent timings (median time per search):


n =   1,000:  linear 9.13 µs    binary 685 ns    (binary search worst case: 10 steps)


n =  10,000:  linear 94.5 µs    binary 1.1 µs    (binary search worst case: 14 steps)


n = 100,000:  linear 1.47 ms    binary 1.29 µs    (binary search worst case: 17 steps)


## O(n log n) vs O(n²): sorting

The question asks to sort an array of `n` numbers from smallest to largest:

- **Insertion sort, O(n²):** moves each number left until it reaches its place. On random input each number moves past about half of the numbers before it, so it is O(n²) in the average and worst cases. On input that is already sorted nothing moves, so the best case is O(n).
- **Merge sort, O(n log n):** splits the array in half, sorts each half, and merges the two sorted halves. It does O(n log n) work whatever the input order.
- **Python's built-in `sorted()`** uses Timsort, a hybrid of merge sort and insertion sort written in C. It is O(n log n) in the average and worst cases, and O(n) in favorable cases such as input that is already sorted.

As `n` doubles, insertion sort's time should grow about 4× and merge sort's a little over 2×. `sorted()` has the same growth rate as merge sort but a much smaller wall-clock time, because it runs as compiled C code instead of Python. The last line shows the favorable case: input that is already sorted.

In [14]:
# solution with O(n^2) time complexity
def insertion_sort(a):
    a = a.copy()
    for i in range(1, len(a)):
        current = a[i]
        j = i - 1
        while j >= 0 and a[j] > current:
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = current
    return a

# solution with O(n log n) time complexity
def merge_sort(a):
    if len(a) <= 1:
        return a.copy()
    mid = len(a) // 2
    left = merge_sort(a[:mid])
    right = merge_sort(a[mid:])
    merged = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged

In [15]:
# fixed seed so the inputs are reproducible
r.seed(42)

print("Illustrative, machine-dependent timings (median time per sort):")
for n in [1000, 2000, 4000]:
    nums = [r.randint(1, 100000000) for i in range(n)]
    assert insertion_sort(nums) == merge_sort(nums) == sorted(nums)

    time_insertion = median_time(insertion_sort, nums)
    time_merge = median_time(merge_sort, nums)
    time_builtin = median_time(sorted, nums)
    print(f"random input,         n = {n:,}:  insertion {format_time(time_insertion)}    merge {format_time(time_merge)}    sorted() {format_time(time_builtin)}")

# favorable case: the same 4,000 numbers, already sorted
already_sorted = sorted(nums)
time_insertion = median_time(insertion_sort, already_sorted)
time_merge = median_time(merge_sort, already_sorted)
time_builtin = median_time(sorted, already_sorted)
print(f"already sorted input, n = {n:,}:  insertion {format_time(time_insertion)}    merge {format_time(time_merge)}    sorted() {format_time(time_builtin)}")

Illustrative, machine-dependent timings (median time per sort):


random input,         n = 1,000:  insertion 13 ms    merge 939 µs    sorted() 21.4 µs


random input,         n = 2,000:  insertion 56.2 ms    merge 2.18 ms    sorted() 48 µs


random input,         n = 4,000:  insertion 219 ms    merge 4.43 ms    sorted() 147 µs


already sorted input, n = 4,000:  insertion 266 µs    merge 3.1 ms    sorted() 14.2 µs
